# Sentiment-Driven Market Analysis  
## Twitter Financial News Sentiment (Kaggle Notebook)

This notebook performs:
- dataset loading and inspection
- pragmatic preprocessing
- exploratory visualizations
- four-model comparison
- evaluation with paper-ready metrics
- error analysis and novelty-oriented hybrid inference

## 1. Objective

Predict financial tweet sentiment using the Hugging Face dataset:

- **Input column:** `text`
- **Target column:** `label`
- **Classes:** `0 = Bearish`, `1 = Bullish`, `2 = Neutral`

The notebook compares:
1. TF-IDF + Logistic Regression  
2. TF-IDF + Linear SVM  
3. Fine-tuned BERT-base  
4. Fine-tuned FinBERT + pragmatic rule-based gating (proposed hybrid)

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn emoji seaborn wordcloud

In [ ]:
import os
import re
import gc
import math
import json
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

import emoji
import torch
from torch.utils.data import Dataset as TorchDataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Load dataset from Hugging Face
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
dataset

In [ ]:
# Convert to pandas for EDA and classical ML
train_df = dataset["train"].to_pandas()
valid_df = dataset["validation"].to_pandas()

print(train_df.head())
print(train_df.columns.tolist())
print(train_df.shape, valid_df.shape)

In [ ]:
LABEL_MAP = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

train_df["label_name"] = train_df["label"].map(LABEL_MAP)
valid_df["label_name"] = valid_df["label"].map(LABEL_MAP)

display(train_df.sample(5, random_state=SEED))

In [ ]:
# Basic dataset checks
print("Train label counts:")
print(train_df["label_name"].value_counts())

print("Validation label counts:")
print(valid_df["label_name"].value_counts())

In [ ]:
# --- Exploratory visualizations ---

plt.figure(figsize=(7, 4))
train_df["label_name"].value_counts().reindex(["Bearish", "Bullish", "Neutral"]).plot(kind="bar")
plt.title("Training Label Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
valid_df["label_name"].value_counts().reindex(["Bearish", "Bullish", "Neutral"]).plot(kind="bar")
plt.title("Validation Label Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

train_df["text_len"] = train_df["text"].astype(str).str.len()
valid_df["text_len"] = valid_df["text"].astype(str).str.len()

plt.figure(figsize=(7, 4))
plt.hist(train_df["text_len"], bins=40)
plt.title("Training Tweet Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(valid_df["text_len"], bins=40)
plt.title("Validation Tweet Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Optional preview of representative examples
for lbl in ["Bearish", "Bullish", "Neutral"]:
    print(f"\n=== {lbl} examples ===")
    examples = train_df[train_df["label_name"] == lbl]["text"].head(3).tolist()
    for ex in examples:
        print("-", ex)

## 2. Pragmatic preprocessing

Two text variants are created:

- **`text_basic`** for classical ML models
- **`text_pragmatic`** for transformer and hybrid analysis

The pragmatic version preserves sentiment-bearing signals such as:
- punctuation emphasis
- repeated letters
- uppercase emphasis
- emojis
- contrast markers (`but`, `however`, `although`)
- negation cues
- hedge markers (`if`, `may`, `could`)

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
MULTI_SPACE_RE = re.compile(r"\s+")
REPEAT_PUNCT_RE = re.compile(r"([!?.,]){2,}")
REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}")
NUMBER_RE = re.compile(r"\b\d+([.,]\d+)?\b")

EMOJI_MAP = {
    "📈": "emoji_bullish",
    "🚀": "emoji_bullish",
    "🔥": "emoji_bullish",
    "💹": "emoji_bullish",
    "📉": "emoji_bearish",
    "💥": "emoji_bearish",
    "😡": "emoji_bearish",
    "😐": "emoji_neutral",
    "🤔": "emoji_hedge",
    "😂": "emoji_sarcasm",
    "🙃": "emoji_sarcasm",
}

CONTRAST_WORDS = {"but", "however", "although", "though", "yet", "while", "despite"}
HEDGE_WORDS = {"if", "may", "might", "could", "would", "possible", "possibly", "seems", "appears", "could"}
NEGATION_WORDS = {"no", "not", "never", "none", "without", "hardly", "neither", "nor"}

POS_WORDS = {
    "beat","beats","beating","surge","surges","up","gain","gains","bullish","upgrade","upgraded",
    "strong","record","profit","growth","buy","outperform","raise","raises","raised","higher","rally",
    "optimistic","good","great","positive","bounce","bull","spike","expands","expansion","improve","improves"
}
NEG_WORDS = {
    "miss","misses","missed","down","loss","losses","bearish","downgrade","downgraded","weak","decline",
    "fall","falls","cut","cuts","lower","short","risk","lawsuit","drop","plunge","negative","bad","concern",
    "warning","warns","warned","slump","bear"
}

def replace_emojis(text: str) -> str:
    for e, token in EMOJI_MAP.items():
        text = text.replace(e, f" {token} ")
    return text

def clean_basic(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r"\1", text)
    text = replace_emojis(text)
    text = text.lower()
    text = NUMBER_RE.sub(" <NUM> ", text)
    text = REPEAT_PUNCT_RE.sub(r" \1 ", text)
    text = REPEAT_CHAR_RE.sub(r"\1\1", text)
    text = re.sub(r"[^a-z<>()!?.,\s_]", " ", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text

def clean_pragmatic(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" <URL> ", text)
    text = MENTION_RE.sub(" <USER> ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)
    text = replace_emojis(text)
    text = NUMBER_RE.sub(" <NUM> ", text)
    text = REPEAT_PUNCT_RE.sub(r" \1 ", text)
    text = REPEAT_CHAR_RE.sub(r"\1\1", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text

def tokenize_simple(text: str):
    return re.findall(r"[A-Za-z_<>]+|[!?.,]", text.lower())

def rule_based_sentiment(text: str):
    tokens = tokenize_simple(text)
    token_set = set(tokens)
    pos_hits = len(token_set & POS_WORDS)
    neg_hits = len(token_set & NEG_WORDS)
    has_contrast = any(w in token_set for w in CONTRAST_WORDS)
    has_hedge = any(w in token_set for w in HEDGE_WORDS)
    has_negation = any(w in token_set for w in NEGATION_WORDS)
    has_sarcasm = ("emoji_sarcasm" in text.lower()) or ('"' in text) or ("..." in text) or ("!? " in text)

    # Very light clause-based heuristic for "X but Y"
    lower = text.lower()
    if any(c in lower for c in CONTRAST_WORDS):
        for c in ["but", "however", "although", "though", "yet"]:
            if c in lower:
                parts = lower.split(c, 1)
                left = parts[0]
                right = parts[1]
                left_pos = sum(w in left for w in POS_WORDS)
                left_neg = sum(w in left for w in NEG_WORDS)
                right_pos = sum(w in right for w in POS_WORDS)
                right_neg = sum(w in right for w in NEG_WORDS)
                if right_neg > right_pos:
                    return "Bearish", 0.90, {"contrast": True, "hedge": has_hedge, "sarcasm": has_sarcasm}
                if right_pos > right_neg:
                    return "Bullish", 0.90, {"contrast": True, "hedge": has_hedge, "sarcasm": has_sarcasm}

    score = pos_hits - neg_hits
    if has_negation:
        score = -score if score != 0 else score

    if has_hedge and abs(score) <= 1:
        return "Neutral", 0.85, {"contrast": has_contrast, "hedge": True, "sarcasm": has_sarcasm}

    if score > 0:
        return "Bullish", min(0.55 + 0.1 * score, 0.95), {"contrast": has_contrast, "hedge": has_hedge, "sarcasm": has_sarcasm}
    if score < 0:
        return "Bearish", min(0.55 + 0.1 * abs(score), 0.95), {"contrast": has_contrast, "hedge": has_hedge, "sarcasm": has_sarcasm}
    return "Neutral", 0.60, {"contrast": has_contrast, "hedge": has_hedge, "sarcasm": has_sarcasm}

train_df["text_basic"] = train_df["text"].apply(clean_basic)
train_df["text_pragmatic"] = train_df["text"].apply(clean_pragmatic)
valid_df["text_basic"] = valid_df["text"].apply(clean_basic)
valid_df["text_pragmatic"] = valid_df["text"].apply(clean_pragmatic)

print(train_df[["text", "text_basic", "text_pragmatic"]].head(3).to_string(index=False))

In [ ]:
# Preprocessing visual sanity check
sample_rows = train_df.sample(5, random_state=SEED)[["text", "text_basic", "text_pragmatic", "label_name"]]
for _, row in sample_rows.iterrows():
    print("\nLABEL:", row["label_name"])
    print("RAW :", row["text"])
    print("BASIC:", row["text_basic"])
    print("PRAG :", row["text_pragmatic"])

## 3. Classical baselines

These models provide strong and interpretable benchmarks:
- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM

They are useful for showing the gain from transformer-based methods.

In [ ]:
X_train = train_df["text_basic"].fillna("").tolist()
y_train = train_df["label"].tolist()
X_valid = valid_df["text_basic"].fillna("").tolist()
y_valid = valid_df["label"].tolist()

def evaluate_predictions(y_true, y_pred, name="model"):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=[LABEL_MAP[i] for i in sorted(LABEL_MAP)], output_dict=True, zero_division=0)
    return {
        "model": name,
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f1_weighted,
        "report": report,
    }

def fit_and_eval_classical(model, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    return pred, evaluate_predictions(y_valid, pred, name=name)

tfidf_logreg = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        min_df=2,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", n_jobs=None))
])

tfidf_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        min_df=2,
        sublinear_tf=True
    )),
    ("clf", LinearSVC(class_weight="balanced"))
])

logreg_pred, logreg_metrics = fit_and_eval_classical(tfidf_logreg, "TFIDF + Logistic Regression")
svm_pred, svm_metrics = fit_and_eval_classical(tfidf_svm, "TFIDF + Linear SVM")

print(logreg_metrics["model"], logreg_metrics["f1_macro"])
print(svm_metrics["model"], svm_metrics["f1_macro"])

## 4. Transformer fine-tuning

Two transformer models are **fully fine-tuned** (all encoder weights unfrozen):

- **FinBERT** (`ProsusAI/finbert`): finance-domain pretrained model, fine-tuned end-to-end
- **DistilBERT-base-uncased**: general-purpose baseline, fine-tuned end-to-end

Both models are trained with a low learning rate (`2e-5`) to avoid catastrophic forgetting.

Key training settings (Kaggle-friendly):
- `max_len = 64` — short tweets, so 64 tokens is sufficient
- `fp16` enabled when GPU is available
- Early stopping on `f1_macro` with patience = 1
- `load_best_model_at_end = True` — always evaluates final checkpoint
- `class_weight` balanced via `Trainer` sampling is not supported natively; 
class imbalance is handled via data preprocessing

> **Why unfreeze?** Frozen encoder experiments showed ~8 pp F1-macro gap vs. the TF-IDF + SVM baseline. 
Full fine-tuning allows the encoder to adapt its representations to financial tweet vocabulary, 
which is expected to close or reverse that gap.

In [ ]:
class TweetDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    DataCollatorWithPadding,
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def train_transformer_ultra_fast(
    train_texts, train_labels, valid_texts, valid_labels,
    output_dir,
    model_name="distilbert-base-uncased",
    epochs=3,
    lr=2e-5,           # low LR for full fine-tuning (avoids catastrophic forgetting)
):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    # All encoder weights are trainable — full fine-tuning
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{model_name}] Trainable parameters: {trainable:,}")

    MAX_LEN = 64
    train_ds = TweetDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
    valid_ds = TweetDataset(valid_texts, valid_labels, tokenizer, max_len=MAX_LEN)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=64 if torch.cuda.is_available() else 8,
        per_device_eval_batch_size=64 if torch.cuda.is_available() else 16,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=0.1,          # 10% warmup helps full fine-tuning stability
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=1,
        logging_steps=100,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred.predictions, eval_pred.label_ids
        preds = np.argmax(logits, axis=1)
        _, _, f1_macro, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        return {"f1_macro": f1_macro}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()

    out = trainer.predict(valid_ds)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=1).numpy()
    pred_labels = np.argmax(probs, axis=1).tolist()
    y_true = list(valid_labels)

    acc = accuracy_score(y_true, pred_labels)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )
    p_wt, r_wt, f1_wt, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="weighted", zero_division=0
    )

    metrics = {
        "model": model_name,
        "accuracy": acc,
        "precision_macro": p_mac,
        "recall_macro": r_mac,
        "f1_macro": f1_mac,
        "precision_weighted": p_wt,
        "recall_weighted": r_wt,
        "f1_weighted": f1_wt,
    }

    print(f"[{model_name}] Acc={acc:.4f}  F1-macro={f1_mac:.4f}  F1-weighted={f1_wt:.4f}")

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "probs": probs,
        "pred_labels": pred_labels,
        "metrics": metrics,
    }

# ── Train DistilBERT (full fine-tuning) ───────────────────────────────────────
bert_results = train_transformer_ultra_fast(
    train_texts=train_df["text_pragmatic"].tolist(),
    train_labels=train_df["label"].tolist(),
    valid_texts=valid_df["text_pragmatic"].tolist(),
    valid_labels=valid_df["label"].tolist(),
    output_dir="./fast_bert_sentiment",
    model_name="distilbert-base-uncased",
    lr=2e-5,
)


In [ ]:
# ── Train FinBERT (full fine-tuning) ─────────────────────────────────────────
finbert_results = train_transformer_ultra_fast(
    model_name="ProsusAI/finbert",
    train_texts=train_df["text_pragmatic"].tolist(),
    train_labels=train_df["label"].tolist(),
    valid_texts=valid_df["text_pragmatic"].tolist(),
    valid_labels=valid_df["label"].tolist(),
    output_dir="./finbert_sentiment",
    epochs=3,
    lr=2e-5,
)

print("FinBERT (fine-tuned) macro-F1:", finbert_results["metrics"]["f1_macro"])


## 5. Proposed hybrid model: FinBERT + pragmatic gating

The proposed novelty layer combines:
- transformer confidence
- contrast/sarcasm/hedging cues
- rule-based override for ambiguous tweets

This is useful for examples where the literal wording and the implied market direction differ.

In [ ]:
# Helper for getting transformer probabilities from trained model
def predict_proba_transformer(trainer, dataset):
    out = trainer.predict(dataset)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    conf = probs.max(axis=1)
    return preds, probs, conf

finbert_tokenizer = finbert_results["tokenizer"]
finbert_trainer = finbert_results["trainer"]
finbert_valid_ds = TweetDataset(
    valid_df["text_pragmatic"].tolist(),
    valid_df["label"].tolist(),
    finbert_tokenizer,
    max_len=128
)

finbert_pred, finbert_probs, finbert_conf = predict_proba_transformer(finbert_trainer, finbert_valid_ds)

ID_TO_LABEL = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
LABEL_TO_ID = {v: k for k, v in ID_TO_LABEL.items()}

def hybrid_override(text, finbert_pred_id, confidence, threshold=0.65):
    rule_label, rule_conf, flags = rule_based_sentiment(text)
    rule_id = LABEL_TO_ID[rule_label]

    # Override when confidence is low or pragmatic cues are strong
    strong_cue = flags["contrast"] or flags["sarcasm"] or flags["hedge"]
    if confidence < threshold or strong_cue:
        return rule_id, {"rule_label": rule_label, "rule_conf": rule_conf, "flags": flags}
    return finbert_pred_id, {"rule_label": rule_label, "rule_conf": rule_conf, "flags": flags}

hybrid_pred = []
hybrid_meta = []

for txt, pred_id, conf in zip(valid_df["text_pragmatic"].tolist(), finbert_pred.tolist(), finbert_conf.tolist()):
    out_id, meta = hybrid_override(txt, pred_id, conf, threshold=0.65)
    hybrid_pred.append(out_id)
    hybrid_meta.append(meta)

hybrid_metrics = evaluate_predictions(y_valid, hybrid_pred, name="FinBERT + Pragmatic Gating")
hybrid_metrics["f1_macro"]

## 6. Results table and paper-ready metrics

The key comparison metric is **Macro F1**, because it treats Bearish, Bullish, and Neutral 
equally regardless of class frequency.

With both transformer models now **fully fine-tuned**, we expect the performance landscape to shift:

- **FinBERT (fine-tuned)** — the primary candidate to surpass the TF-IDF baseline; 
its finance-domain pretraining + gradient flow through all encoder layers should yield 
strong domain-aligned representations.
- **DistilBERT (fine-tuned)** — a general-purpose model; likely improves significantly over 
its frozen counterpart but may still trail FinBERT due to lack of domain-specific pretraining.
- **TF-IDF + LinearSVM** — remains the interpretable, lightweight baseline. 
Strong when fine-tuned transformers are not available or compute is limited.
- **FinBERT + Pragmatic Gating** — hybrid confidence-override layer on top of fine-tuned FinBERT; 
expected to improve precision on hedged and contrastive tweets.

Results are sorted by **Macro F1** descending. 
The best model across all metrics is automatically selected and saved in Section 11.


In [ ]:
# ── Collect all metrics ──────────────────────────────────────────────────────
bert_m    = bert_results["metrics"]
finbert_m = finbert_results["metrics"]

results = pd.DataFrame([
    {
        "Model":             logreg_metrics["model"],
        "Accuracy":          logreg_metrics["accuracy"],
        "Precision (Macro)": logreg_metrics["precision_macro"],
        "Recall (Macro)":    logreg_metrics["recall_macro"],
        "F1 (Macro)":        logreg_metrics["f1_macro"],
        "F1 (Weighted)":     logreg_metrics["f1_weighted"],
    },
    {
        "Model":             svm_metrics["model"],
        "Accuracy":          svm_metrics["accuracy"],
        "Precision (Macro)": svm_metrics["precision_macro"],
        "Recall (Macro)":    svm_metrics["recall_macro"],
        "F1 (Macro)":        svm_metrics["f1_macro"],
        "F1 (Weighted)":     svm_metrics["f1_weighted"],
    },
    {
        "Model":             "DistilBERT-base-uncased (fine-tuned)",
        "Accuracy":          bert_m["accuracy"],
        "Precision (Macro)": bert_m["precision_macro"],
        "Recall (Macro)":    bert_m["recall_macro"],
        "F1 (Macro)":        bert_m["f1_macro"],
        "F1 (Weighted)":     bert_m["f1_weighted"],
    },
    {
        "Model":             "FinBERT (fine-tuned)",
        "Accuracy":          finbert_m["accuracy"],
        "Precision (Macro)": finbert_m["precision_macro"],
        "Recall (Macro)":    finbert_m["recall_macro"],
        "F1 (Macro)":        finbert_m["f1_macro"],
        "F1 (Weighted)":     finbert_m["f1_weighted"],
    },
    {
        "Model":             hybrid_metrics["model"],
        "Accuracy":          hybrid_metrics["accuracy"],
        "Precision (Macro)": hybrid_metrics["precision_macro"],
        "Recall (Macro)":    hybrid_metrics["recall_macro"],
        "F1 (Macro)":        hybrid_metrics["f1_macro"],
        "F1 (Weighted)":     hybrid_metrics["f1_weighted"],
    },
]).sort_values("F1 (Macro)", ascending=False).reset_index(drop=True)

num_cols = [c for c in results.columns if c != "Model"]
display(
    results.style
        .format({c: "{:.4f}" for c in num_cols})
        .highlight_max(subset=num_cols, color="lightgreen")
        .set_caption("Table 1 — Model comparison (sorted by Macro F1 ↓)")
)


In [ ]:
# Save results for paper tables
results.to_csv("model_comparison_results.csv", index=False)
results

In [ ]:
# ── HD global plot style — applies to ALL visualizations below ──────────────
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams.update({
    # Font
    "font.family":        "DejaVu Sans",
    "font.size":          12,
    "axes.titlesize":     14,
    "axes.titleweight":   "bold",
    "axes.titlepad":      14,
    "axes.labelsize":     12,
    "axes.labelpad":      8,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "legend.fontsize":    10,
    "legend.title_fontsize": 10,
    # Lines & patches
    "lines.linewidth":    2.0,
    "patch.linewidth":    0.6,
    # Grid
    "axes.grid":          True,
    "grid.alpha":         0.28,
    "grid.linestyle":     "--",
    "grid.linewidth":     0.7,
    # Spines
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    # Output quality
    "figure.dpi":         150,      # screen preview
    "savefig.dpi":        300,      # saved files — HD
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.15,
    # Anti-aliasing
    "text.antialiased":   True,
    "lines.antialiased":  True,
    "patch.antialiased":  True,
    "figure.autolayout":  False,
})
sns.set_theme(style="whitegrid", font_scale=1.1)
sns.set_palette("tab10")

# Palette used across all charts
MODEL_COLORS = {
    "TF-IDF + Logistic Regression":        "#4C72B0",
    "TFIDF + Logistic Regression":          "#4C72B0",
    "TFIDF + Linear SVM":                  "#DD8452",
    "TF-IDF + Linear SVM":                 "#DD8452",
    "DistilBERT-base-uncased (fine-tuned)": "#55A868",
    "FinBERT (fine-tuned)":                "#C44E52",
    "FinBERT + Pragmatic Gating":           "#8172B2",
}
PALETTE5 = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
print("HD plot style applied. savefig.dpi = 300")


## 7. Visualizing model comparison

This section provides a full visual evaluation suite:

1. **Multi-metric grouped bar chart** — all 5 metrics side-by-side per model
2. **Per-class F1 heatmap** — Bearish / Bullish / Neutral F1 breakdown per model
3. **Radar (spider) chart** — holistic multi-metric profile per model
4. **Confusion matrices** — per-model prediction error patterns
5. **ROC curves (One-vs-Rest)** — AUC per class for all probabilistic models
6. **Precision-Recall curves** — Average Precision (AP) per class
7. **Calibration (reliability) diagrams** — how well confidence scores match actual accuracy


In [ ]:
# ── 7.1  Multi-metric grouped bar chart ─────────────────────────────────────
METRICS     = ["Accuracy","Precision (Macro)","Recall (Macro)","F1 (Macro)","F1 (Weighted)"]
METRIC_LBLS = ["Accuracy","Precision\n(Macro)","Recall\n(Macro)","F1\n(Macro)","F1\n(Weighted)"]
METRIC_COLS = ["#3274A1","#E1812C","#3A923A","#C03D3E","#9372B2"]
models  = results["Model"].tolist()
x       = np.arange(len(models))
bar_w   = 0.13
n_met   = len(METRICS)

fig, ax = plt.subplots(figsize=(16, 6))
for i, (metric, mlabel, color) in enumerate(zip(METRICS, METRIC_LBLS, METRIC_COLS)):
    vals   = results[metric].values
    offset = (i - n_met / 2 + 0.5) * bar_w
    bars   = ax.bar(x + offset, vals, bar_w, label=mlabel,
                    color=color, alpha=0.88, edgecolor="white", linewidth=0.5,
                    zorder=3)
    # value labels on top of each bar
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                f"{h:.3f}", ha="center", va="bottom",
                fontsize=7, color="#333333")

ax.set_xticks(x)
short_names = [
    m.replace("TFIDF + ","TF-IDF+").replace(" (fine-tuned)","\n(fine-tuned)")
     .replace(" + Pragmatic Gating","\n+Pragmatic Gating")
    for m in models
]
ax.set_xticklabels(short_names, fontsize=9.5, linespacing=1.4)
ax.set_ylim(0.55, 1.04)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Model Comparison — All Metrics", fontsize=15)
ax.legend(loc="lower right", framealpha=0.85, ncol=5,
          fontsize=9, handlelength=1.2, columnspacing=0.8)
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout(pad=2.0)
plt.savefig("viz_01_multi_metric_bar.png")
plt.show()
print("Saved → viz_01_multi_metric_bar.png  (300 dpi)")


In [ ]:
# ── 7.2  Per-class F1 heatmap ────────────────────────────────────────────────
from sklearn.metrics import classification_report

model_preds = [
    ("TF-IDF + LogReg",          y_valid, logreg_pred),
    ("TF-IDF + LinearSVM",       y_valid, svm_pred),
    ("DistilBERT\n(fine-tuned)", y_valid, bert_results["pred_labels"]),
    ("FinBERT\n(fine-tuned)",    y_valid, finbert_results["pred_labels"]),
    ("FinBERT\n+Gating",         y_valid, hybrid_pred),
]
class_names = [LABEL_MAP[i] for i in [0,1,2]]

f1_matrix = []
for name, y_t, y_p in model_preds:
    rep = classification_report(y_t, y_p, target_names=class_names,
                                output_dict=True, zero_division=0)
    f1_matrix.append([rep[c]["f1-score"] for c in class_names])

f1_df = pd.DataFrame(f1_matrix,
                     index=[n for n,_,_ in model_preds],
                     columns=class_names)

fig, ax = plt.subplots(figsize=(8, 5.5))
hm = sns.heatmap(
    f1_df, annot=True, fmt=".3f", cmap="YlGn",
    vmin=0.50, vmax=1.0, linewidths=1.0, linecolor="white",
    annot_kws={"fontsize": 13, "fontweight": "bold"},
    cbar_kws={"shrink": 0.82, "label": "F1 Score"},
    ax=ax,
)
hm.set_title("Per-class F1 Score by Model", fontsize=15, pad=14)
hm.set_xlabel("Sentiment Class", fontsize=12, labelpad=10)
hm.set_ylabel("", fontsize=12)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=10, rotation=0)
fig.tight_layout(pad=2.0)
plt.savefig("viz_02_perclass_f1_heatmap.png")
plt.show()
print("Saved → viz_02_perclass_f1_heatmap.png  (300 dpi)")


In [ ]:
# ── 7.3  Radar (spider) chart ────────────────────────────────────────────────
RADAR_METRICS = ["Accuracy","Precision (Macro)","Recall (Macro)","F1 (Macro)","F1 (Weighted)"]
RADAR_LBLS    = ["Accuracy","Precision\n(Macro)","Recall\n(Macro)","F1\n(Macro)","F1\n(Weighted)"]

N      = len(RADAR_METRICS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_LBLS, fontsize=11, linespacing=1.5)
ax.set_ylim(0.55, 1.0)
ax.set_yticks([0.60, 0.70, 0.80, 0.90, 1.00])
ax.set_yticklabels(["0.60","0.70","0.80","0.90","1.00"],
                   fontsize=8.5, color="#555555")
ax.yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.5)
ax.xaxis.grid(True, linestyle="-",  linewidth=0.5, alpha=0.3)
ax.spines["polar"].set_visible(False)

for name, color in zip(results["Model"].tolist(), PALETTE5):
    row  = results[results["Model"]==name].iloc[0]
    vals = [row[m] for m in RADAR_METRICS] + [row[RADAR_METRICS[0]]]
    short = (name.replace("TFIDF + ","TF-IDF+")
                 .replace(" (fine-tuned)"," (ft)")
                 .replace(" + Pragmatic Gating","+Gating"))
    ax.plot(angles, vals, linewidth=2.2, color=color, label=short)
    ax.fill(angles, vals, alpha=0.07, color=color)
    # Label the highest point of each model
    peak_idx = int(np.argmax(vals[:-1]))
    ax.annotate(
        f"{vals[peak_idx]:.3f}",
        xy=(angles[peak_idx], vals[peak_idx]),
        fontsize=7.5, color=color, fontweight="bold",
        xytext=(6, 4), textcoords="offset points",
    )

ax.set_title("Model Radar — All Metrics", fontsize=15, pad=22)
ax.legend(loc="upper right", bbox_to_anchor=(1.42, 1.18),
          fontsize=9.5, framealpha=0.88,
          edgecolor="#cccccc", handlelength=1.5)
fig.tight_layout(pad=2.5)
plt.savefig("viz_03_radar.png")
plt.show()
print("Saved → viz_03_radar.png  (300 dpi)")


In [ ]:
# ── 7.4  Confusion matrices ──────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

cm_models = [
    ("TF-IDF + Logistic Regression",          y_valid, logreg_pred),
    ("TF-IDF + Linear SVM",                   y_valid, svm_pred),
    ("DistilBERT-base-uncased (fine-tuned)",   y_valid, bert_results["pred_labels"]),
    ("FinBERT (fine-tuned)",                   y_valid, finbert_results["pred_labels"]),
    ("FinBERT + Pragmatic Gating",             y_valid, hybrid_pred),
]
class_names = [LABEL_MAP[i] for i in [0,1,2]]

fig, axes = plt.subplots(1, 5, figsize=(26, 5.2))
for ax, (name, y_t, y_p) in zip(axes, cm_models):
    cm = confusion_matrix(y_t, y_p, labels=[0,1,2])
    # Percentage annotation: count + %
    total  = cm.sum()
    annots = np.array([[f"{v}\n({v/total*100:.1f}%)" for v in row] for row in cm])
    sns.heatmap(
        cm, annot=annots, fmt="", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.8, linecolor="white",
        cbar=False,
        annot_kws={"fontsize": 9.5},
        ax=ax,
    )
    short = name.replace(" (fine-tuned)","\n(fine-tuned)").replace(" + Pragmatic Gating","\n+Pragmatic Gating")
    ax.set_title(short, fontsize=10.5, pad=10)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True" if ax is axes[0] else "", fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=9)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)

fig.suptitle("Confusion Matrices — All Models", fontsize=15, y=1.02)
fig.tight_layout(pad=2.0)
plt.savefig("viz_04_confusion_matrices.png")
plt.show()
print("Saved → viz_04_confusion_matrices.png  (300 dpi)")


In [ ]:
# ── 7.5  ROC curves (One-vs-Rest) ────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_valid, classes=[0,1,2])

def svm_probs(pipeline, X):
    scores = pipeline.decision_function(X)
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

prob_models = [
    ("TF-IDF + LogReg",           "#4C72B0", tfidf_logreg.predict_proba(X_valid)),
    ("TF-IDF + LinearSVM",        "#DD8452", svm_probs(tfidf_svm, X_valid)),
    ("DistilBERT (fine-tuned)",   "#55A868", bert_results["probs"]),
    ("FinBERT (fine-tuned)",      "#C44E52", finbert_results["probs"]),
    ("FinBERT + Gating",          "#8172B2", finbert_probs),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharey=True)
for cls_idx, (cls_name, ax) in enumerate(zip(class_names, axes)):
    for name, color, probs in prob_models:
        fpr, tpr, _ = roc_curve(y_bin[:, cls_idx], probs[:, cls_idx])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2.0, color=color,
                label=f"{name}  (AUC = {roc_auc:.3f})")
    ax.plot([0,1],[0,1], color="#aaaaaa", lw=1.2, linestyle="--", label="Random")
    ax.fill_between([0,1],[0,1], alpha=0.04, color="grey")
    ax.set_title(f"ROC — {cls_name}", fontsize=13)
    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.legend(fontsize=7.5, loc="lower right",
              framealpha=0.88, edgecolor="#cccccc")
    ax.tick_params(labelsize=10)

axes[0].set_ylabel("True Positive Rate", fontsize=11)
fig.suptitle("ROC Curves — One-vs-Rest per Sentiment Class", fontsize=15, y=1.01)
fig.tight_layout(pad=2.2)
plt.savefig("viz_05_roc_curves.png")
plt.show()
print("Saved → viz_05_roc_curves.png  (300 dpi)")


In [ ]:
# ── 7.6  Precision-Recall curves ─────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharey=True)
for cls_idx, (cls_name, ax) in enumerate(zip(class_names, axes)):
    for name, color, probs in prob_models:
        prec, rec, _ = precision_recall_curve(y_bin[:, cls_idx], probs[:, cls_idx])
        ap            = average_precision_score(y_bin[:, cls_idx], probs[:, cls_idx])
        ax.plot(rec, prec, lw=2.0, color=color,
                label=f"{name}  (AP = {ap:.3f})")
    baseline = y_bin[:, cls_idx].mean()
    ax.axhline(baseline, color="#999999", linestyle=":", lw=1.5,
               label=f"No-skill baseline ({baseline:.2f})")
    ax.set_title(f"Precision-Recall — {cls_name}", fontsize=13)
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.legend(fontsize=7.5, loc="lower left",
              framealpha=0.88, edgecolor="#cccccc")
    ax.tick_params(labelsize=10)

axes[0].set_ylabel("Precision", fontsize=11)
fig.suptitle("Precision-Recall Curves — One-vs-Rest per Sentiment Class", fontsize=15, y=1.01)
fig.tight_layout(pad=2.2)
plt.savefig("viz_06_pr_curves.png")
plt.show()
print("Saved → viz_06_pr_curves.png  (300 dpi)")


In [ ]:
# ── 7.7  Calibration (reliability) diagrams ──────────────────────────────────
from sklearn.calibration import calibration_curve

cal_models = [
    ("TF-IDF + LogReg",         "#4C72B0", tfidf_logreg.predict_proba(X_valid)),
    ("DistilBERT (fine-tuned)", "#55A868", bert_results["probs"]),
    ("FinBERT (fine-tuned)",    "#C44E52", finbert_results["probs"]),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharey=True)
for cls_idx, (cls_name, ax) in enumerate(zip(class_names, axes)):
    for name, color, probs in cal_models:
        prob_true, prob_pred = calibration_curve(
            y_bin[:, cls_idx], probs[:, cls_idx],
            n_bins=10, strategy="quantile"
        )
        ax.plot(prob_pred, prob_true, marker="o", markersize=5,
                lw=2.0, color=color, label=name)
        ax.fill_between(prob_pred, prob_pred, prob_true,
                        alpha=0.08, color=color)
    ax.plot([0,1],[0,1], color="#888888", lw=1.5, linestyle="--",
            label="Perfect calibration")
    ax.set_title(f"Calibration — {cls_name}", fontsize=13)
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    ax.legend(fontsize=9, loc="upper left",
              framealpha=0.88, edgecolor="#cccccc")
    ax.tick_params(labelsize=10)

axes[0].set_ylabel("Fraction of Positives", fontsize=11)
fig.suptitle("Calibration Diagrams — Reliability per Sentiment Class", fontsize=15, y=1.01)
fig.tight_layout(pad=2.2)
plt.savefig("viz_07_calibration.png")
plt.show()
print("Saved → viz_07_calibration.png  (300 dpi)")


## 8. Error analysis

Inspect the tweets the proposed model misclassified.  
This section supports the discussion of linguistic edge cases such as:
- sarcasm
- hedging
- contrastive clauses
- mixed sentiment
- market jargon

In [ ]:
hybrid_pred_names = [ID_TO_LABEL[i] for i in hybrid_pred]
true_names = [ID_TO_LABEL[i] for i in y_valid]

error_df = valid_df.copy()
error_df["true_label"] = true_names
error_df["pred_label"] = hybrid_pred_names
error_df["correct"] = error_df["true_label"] == error_df["pred_label"]
error_df["finbert_conf"] = finbert_conf
error_df["rule_label"] = [m["rule_label"] for m in hybrid_meta]
error_df["rule_conf"] = [m["rule_conf"] for m in hybrid_meta]

misclassified = error_df[~error_df["correct"]][["text", "true_label", "pred_label", "finbert_conf", "rule_label", "rule_conf"]]
display(misclassified.head(20))

In [ ]:
# Save misclassified samples for discussion in the paper
misclassified.to_csv("hybrid_misclassified_samples.csv", index=False)
print("Saved:", os.path.abspath("model_comparison_results.csv"))
print("Saved:", os.path.abspath("hybrid_misclassified_samples.csv"))

## 9. Paper-ready narrative points

### Result summary

With full fine-tuning enabled, the transformer models are expected to surpass the TF-IDF baselines. 
**FinBERT (fine-tuned)** is the primary candidate for best overall model, given its domain-specific 
pretraining on financial corpora combined with end-to-end gradient updates on the tweet classification task.

The best model is automatically identified by Macro F1 and saved in Section 11.

### Why full fine-tuning matters

- **Frozen encoders** only train the classification head. The encoder's pretrained representations 
may not align with the financial-tweet distribution, leaving ~8 pp of Macro F1 on the table.
- **Full fine-tuning** propagates gradients through all encoder layers, allowing the model to 
adapt attention patterns to domain-specific tokens (*beat*, *miss*, *downgrade*, *rally*).
- A **low learning rate (2e-5)** and **10% warmup** prevent catastrophic forgetting of the 
pretrained linguistic knowledge.

### Novelty

- The **FinBERT + Pragmatic Gating** hybrid applies a rule-based override on top of the fine-tuned 
FinBERT when model confidence is below threshold (0.65) or when contrast/sarcasm/hedging cues are detected.
- This separates decisions the model is confident about from decisions driven by pragmatic cues, 
which is useful in financial NLP where exact wording matters.

### Visualization suite

- **Multi-metric bar** — overall performance comparison
- **Per-class F1 heatmap** — reveals which classes each model struggles with (Bearish is typically hardest)
- **Radar chart** — holistic multi-metric fingerprint of each model
- **Confusion matrices** — per-model error patterns
- **ROC curves (OvR)** — class-level discriminative ability; AUC is threshold-independent
- **PR curves (OvR)** — Average Precision highlights minority-class precision trade-offs
- **Calibration diagrams** — whether predicted probabilities reflect true class frequencies

### Limitations and future work

- Pragmatic lexicon is manually curated; learned confidence calibration would improve the hybrid.
- Dataset is English-only; cross-lingual generalisation untested.
- `max_len=64` may truncate a small number of longer tweets; 128 would be safer with more compute.


## 10. Next extension ideas

If you want to make the paper even stronger, consider:

- **Ablation study**: compare frozen vs. fine-tuned FinBERT side-by-side (now easy — just re-run with the freeze block)
- **Statistical significance**: McNemar's test between the best model and each baseline
- **SHAP for classical models**: explain which ngrams drive Bearish/Bullish/Neutral predictions
- **Attention visualisation**: BertViz or `transformers-interpret` on the fine-tuned FinBERT
- **Hyperparameter sweep**: learning rate, batch size, warmup for FinBERT via Optuna
- **Ensemble**: average FinBERT + Pragmatic Gating probabilities with TF-IDF + SVM scores
- **Deployment**: the best model is saved in Section 11 and ready for the Streamlit app


## 11. Saving the best model for Streamlit deployment

### GitHub file-size constraints

| Model type | Approx size | GitHub direct upload | Solution |
|---|---|---|---|
| TF-IDF + SVM / LogReg (joblib) | ~3–5 MB | ✅ Yes | Direct commit |
| DistilBERT (fine-tuned) | ~250 MB | ❌ >100 MB limit | Hugging Face Hub |
| FinBERT (fine-tuned) | ~400 MB | ❌ >100 MB limit | Hugging Face Hub |

### Strategy

1. **Best model = classical** → save with `joblib` → commit directly to GitHub repo
2. **Best model = transformer** → push to **Hugging Face Hub** (free) → Streamlit loads from hub URL at runtime

Both paths produce a `deployment_config.json` that `app.py` reads to know exactly how to load the model. 
No code changes to the Streamlit app are needed when switching between classical and transformer winners.

### Streamlit repo structure
```
your-repo/
├── app.py
├── requirements.txt
├── deployment_config.json     ← generated below
├── label_map.json             ← generated below
└── best_model_pipeline.joblib ← only if best model is classical
```


In [ ]:
import joblib, json, os, shutil

# ── 1. Best model by Macro F1 (results is sorted descending) ─────────────────
best_row  = results.iloc[0]
best_name = best_row["Model"]
best_f1   = best_row["F1 (Macro)"]
print(f"Best model : {best_name}")
print(f"Macro F1   : {best_f1:.4f}")
print(f"Accuracy   : {best_row['Accuracy']:.4f}")

# ── 2. Registry: display name → (type, object) ───────────────────────────────
model_registry = {
    "TFIDF + Logistic Regression":           ("classical",    tfidf_logreg, None),
    "TFIDF + Linear SVM":                    ("classical",    tfidf_svm,    None),
    "DistilBERT-base-uncased (fine-tuned)":  ("transformer",  bert_results, "distilbert-base-uncased"),
    "FinBERT (fine-tuned)":                  ("transformer",  finbert_results, "ProsusAI/finbert"),
    "FinBERT + Pragmatic Gating":            ("transformer",  finbert_results, "ProsusAI/finbert"),
}
model_type, model_obj, base_model_id = model_registry[best_name]

# ── 3a. Classical → joblib (GitHub-safe, ~3-5 MB) ────────────────────────────
JOBLIB_PATH = "best_model_pipeline.joblib"
LABEL_PATH  = "label_map.json"
CFG_PATH    = "deployment_config.json"

if model_type == "classical":
    joblib.dump(model_obj, JOBLIB_PATH, compress=("zlib", 3))
    size_mb = os.path.getsize(JOBLIB_PATH) / 1e6
    print(f"\n[Classical] Saved → {JOBLIB_PATH}  ({size_mb:.1f} MB)  ✅ GitHub-safe")

    cfg = {
        "model_type":       "classical",
        "model_file":       JOBLIB_PATH,
        "preprocessing":    "clean_basic",
        "best_model_name":  best_name,
        "f1_macro":         round(best_f1, 4),
        "accuracy":         round(best_row["Accuracy"], 4),
        "f1_weighted":      round(best_row["F1 (Weighted)"], 4),
    }

# ── 3b. Transformer → save locally + instructions to push to HF Hub ──────────
else:
    TRANS_DIR = "best_transformer_model"
    os.makedirs(TRANS_DIR, exist_ok=True)
    model_obj["trainer"].save_model(TRANS_DIR)
    model_obj["tokenizer"].save_pretrained(TRANS_DIR)
    print(f"\n[Transformer] Saved locally → {TRANS_DIR}/")
    print("  ⚠️  Size ~250-400 MB — too large for GitHub direct commit.")
    print("  ➜  Push to Hugging Face Hub (free):")
    print("     1. pip install huggingface_hub")
    print("     2. huggingface-cli login")
    print(f"     3. model_obj['model'].push_to_hub('your-username/twitter-sentiment')")
    print(f"     4. model_obj['tokenizer'].push_to_hub('your-username/twitter-sentiment')")
    print("  ➜  Then update deployment_config.json hub_model_id and commit that to GitHub.")

    cfg = {
        "model_type":       "transformer",
        "hub_model_id":     "YOUR_HF_USERNAME/twitter-sentiment",  # update after push
        "base_model_id":    base_model_id,
        "preprocessing":    "clean_basic",
        "best_model_name":  best_name,
        "f1_macro":         round(best_f1, 4),
        "accuracy":         round(best_row["Accuracy"], 4),
        "f1_weighted":      round(best_row["F1 (Weighted)"], 4),
        "max_length":       64,
    }

# ── 4. Save shared artefacts ──────────────────────────────────────────────────
with open(LABEL_PATH, "w") as f:
    json.dump({str(k): v for k, v in LABEL_MAP.items()}, f, indent=2)

with open(CFG_PATH, "w") as f:
    json.dump(cfg, f, indent=2)

print(f"\nSaved → {LABEL_PATH}")
print(f"Saved → {CFG_PATH}")
print("\ndeployment_config.json:")
print(json.dumps(cfg, indent=2))


In [ ]:
# ── 5. Smoke-test the saved artefacts ────────────────────────────────────────
test_tweets = [
    "$AAPL beats Q3 earnings — revenue surges 12% YoY 📈",
    "Markets plunge as Fed signals more aggressive rate hikes 📉",
    "Goldman Sachs maintains Q4 outlook with no revision to estimates",
]

with open(CFG_PATH) as f:
    loaded_cfg = json.load(f)

if loaded_cfg["model_type"] == "classical":
    loaded_model = joblib.load(loaded_cfg["model_file"])
    cleaned      = [clean_basic(t) for t in test_tweets]
    pred_ids     = loaded_model.predict(cleaned)
    pred_labels  = [LABEL_MAP[p] for p in pred_ids]

else:  # transformer — loads from local save directory
    from transformers import pipeline as hf_pipeline
    clf = hf_pipeline(
        "text-classification",
        model=TRANS_DIR,
        tokenizer=TRANS_DIR,
        device=0 if torch.cuda.is_available() else -1,
        truncation=True, max_length=loaded_cfg["max_length"],
    )
    id2label    = {str(k): v for k, v in LABEL_MAP.items()}
    raw         = clf(test_tweets)
    pred_labels = [id2label[r["label"].split("_")[1]] for r in raw]

print("Smoke-test results:")
for tweet, label in zip(test_tweets, pred_labels):
    print(f"  [{label:>8}]  {tweet}")
print("\n✅ Deployment artefacts verified.")
